# Time Series with `pandas`

**Credit:** Notebook created by Eni Mustafaraj, loosely based on Chapter 10 of "Python for Data Analysis" by Wes McKinney.


**Table of Contents**  
1. [Time series basics](#sec2)
2. [Indexing and Selection](#sec3)
3. [Resampling and Frequency Conversion](#sec4)
4. [Wikipedia Revision Timeseries](#sec5)

**Introduction**

What is a time series? Anything that is **observed or measured** at many points in time.  
There are:

1. fixed-frequency time series (data points occur at regular intervals)
2. irregular time series (no fixed offset between units)

**RUNNING Example:** The example time series in this notebook is the one that shows the history of revisions made in the page of the actress [Rose McGowan](https://en.wikipedia.org/wiki/Rose_McGowan). Ms. McGowan was one of the names mentioned in the context of the Harvey Weinstein sexual misconduct allegations in October 2017. Our data contains a list of usernames and timestamps (as strings), stored as a JSON file (which is in the folder of this notebook).

In [ ]:
import json
with open('mcgowan_timestamps.json', 'r') as inputFile:
    usersAndDates = json.load(inputFile)
    
print("length of revisions:", len(usersAndDates))

In [ ]:
# look at a few elements
usersAndDates[:5]

<a id="sec1"></a>

In [ ]:
import pandas as pd
from pandas import Series, DataFrame

<a id="sec2"></a>
## 1. Time series basics

The most basic kind of time series in pandas is a `Series` indexed by timestamps. One can use as index a list of `datetime` objects created in Python, for example:

In [ ]:
from datetime import datetime
# create a list of 6 date objects
dates = [datetime(2021, 1, 2), datetime(2021, 1, 5), datetime(2021, 1, 7),
         datetime(2021, 1, 8), datetime(2021, 1, 10), datetime(2021, 1, 12)]

We can supply the created list for the `index` parameter:

In [ ]:
import numpy as np
ts = Series(np.random.randn(6), index=dates)
ts

`pandas` creates a new data type for the index column, called `DateTimeIndex`:

In [ ]:
type(ts.index)

To see the difference, let's create a simple series object that gets its index automatically from pandas:

In [ ]:
s = Series(np.random.randn(6))
s

In [ ]:
type(s.index)

### Converting the Wiki data into a time series
Let's look now at how to create a `Series` where time is an index for the Wikipedia data we loaded at the start of the notebook. 

We will use the function `zip` to create two separate columns: one for the timestamps and one for the usernames.
The function `zip` in function can be used in two ways: to zip two sequences into one, and to unzip a sequence into two or more sequences. Below are some examples.

In [ ]:
# The unzipping feature uses the operator *

pairs = [(1, 'a'), (2, 'b'), (3, 'c'), (4, 'd')]
numbers, letters = zip(*pairs)
print(numbers)
print(letters)

In [ ]:
# we can now zip these two sequences again, but in the reverse order

list(zip(letters, numbers))

Now that we now how the `zip` function works, we can use it to unzip the given data into two separate lists:

In [ ]:
usernames, timestamps = zip(*usersAndDates)
usernames[:3], timestamps[:3]

The timestamps are as strings, meanwhile, our example above for creating the timeseries, used datetime objects. It turns out, pandas has its own function that takes a string and turns it into a datetimeindex object.

In [ ]:
pd.to_datetime(['2017-10-21 09:32:02'])

Now that we know this, we can create our timeseries of Wikipedia revisions:

In [ ]:
# time series for revisions
tsRevWiki = Series(usernames, index=pd.to_datetime(timestamps))
tsRevWiki.head(10)

As we saw above in the made-up example with random values, the timestamp column is converted into a `DatetimeIndex` object by `pandas`:

In [ ]:
tsRevWiki.index

However, the values inside this index are `Timestamp` instances:

In [ ]:
tsRevWiki.index[0]

A `Timestamp` instance has many more methods (useful for analysis) than `datetime` instances:

In [ ]:
print(dir(tsRevWiki.index[0]))

For example, notice properties such as `is_month_start`, `weekofyear`, etc.

<a id="sec3"></a>
## 2. Indexing and Selection
In a `Series`, if we use the indices 0 to n-1, we access the variable (the column of the Series), not the index:

In [ ]:
tsRevWiki.iloc[2]

Thus, to access the "index" value, we use the `index` attribute:

In [ ]:
tsRevWiki.index[2]

If this index value is stored in a variable, it can be used for indexing instead of numbers 0 to n-1.  
That is, instead of accessing the values in the column through numbers 0 to n-1, we can access them through their index value:

In [ ]:
moment = tsRevWiki.index[2]
print("moment:", moment)
print("user:", tsRevWiki[moment])

### Accessing values via dates

As we just saw, the primary benefit of a DatetimeIndex is that we can use datetime strings to access data from the series. These can be different valid strings. Let's start with a date (year, month, day):

In [ ]:
tsRevWiki['2017-10-17']

This works with incomplete dates as well. Here is with year and month:

In [ ]:
tsRevWiki['2017-10'].head(10) # showing only the first 10 values, because too many

This result was long, we can check the size of the subseries:

In [ ]:
tsRevWiki['2017-10'].count()

What about the entire year of 2017:

In [ ]:
tsRevWiki['2017'].count()

**Conclusion:** pandas provides a powerful way to query a times series through string date values.

### Exercise: How to find the number of edits by year? 
We will learn a better method later in this notebook, but here is one that you can do too.  
**TIP:** Try to unpack the expressions to see the role of each method.

In [ ]:
# find the first year and last year of revisions
minR = tsRevWiki.index.min().year      # find min value, get its year
maxR = tsRevWiki.index.max().year      # find max value, get its year

print(minR, maxR)

In [ ]:
# create a range of years and with a for loop to access the Series

for year in range(minR, maxR+1):
    print(year, tsRevWiki[str(year)].count())

<a id="sec4"></a>

## 3. Resampling and Frequency Conversion
_Resampling_ refers to the process of converting a time series from one frequency to another. Aggregating higher frequency data  
to lower frequency is called _downsampling_ while converting lower frequency to higher frequency is called _upsampling_ .

For regular dataseries (or fixed-frequency) dataseries, the frequency is the interval of time between two measurements. For example: measuring the temperature every 6 hours; the blood pressure every week; the stock price at the end of the day, and so on. 

For irregular data series, like the Wikipedia revisions, which are mostly random (at the will of the editors, though not entirely random, they depend on events in the real world), there is no fixed frequency. In order to study the timeseries though, we might want to use a chosen frequency unit: a day, a week, a month, a year.

Before learning how to do that, let's talk about `date_range` and frequency syntax.

### The `date_range` function

The statements below create a timeseries of random numbers, one number for each day (that is what `freq='D'` means), for a period of 100 days in total.

In [ ]:
from numpy.random import randn
drange = pd.date_range('1/1/2000', periods=100, freq='D')
ts = Series(randn(len(drange)), index=drange)
ts.head(10) 

We can indeed check that we have 100 data points:

In [ ]:
ts.count()

There are many values that the parameter `freq` can take, here are some more examples:

In [ ]:
# frequency is 3 days
pd.date_range("01/01/17", periods=31, freq="3D")

As we can see, 31 days were created, all 3 days apart.

In [ ]:
# make the "frequency step" to be 90 minutes
pd.date_range("01/01/17 00:00", periods=10, freq="90min")

If we don't provide a value for the frequency parameter, by default is 'D' for one day. The `date_range` function takes a start and end date, like below:

In [ ]:
pd.date_range('10/10/2019', '10/20/2019')

Below is a table that contains some of the values that can be passed to the `freq` parameter. For a complete version of this table, you should visit [this documentation page](https://pandas.pydata.org/docs/user_guide/timeseries.html#dateoffset-objects) and then scroll down a bit to see it.

<img src="frequency.png" width=600>

### A resampling example

For the timeseries `ts` we created above with random numbers, let's see what resampling does.

Because the values were "recorded" daily, we will ask for a resampling based on a month:

In [ ]:
ts.resample("ME").mean()

Since we are changing the data to go from daily values to a monthly value, once we resample, we have to specify what kind of value we want (mean, sum, max, min, some other function, etc.)

In [ ]:
# let's find the sum for the weekly sample (all data collected in one week)
ts.resample("W").sum()

<a id="sec5"></a>

## 4. Wikipedia Revisions Timeseries

As a reminder, we created a series `tsRevWiki` earlier in this notebook, let's look at it again:

In [ ]:
tsRevWiki.head(10)

How big is this series:

In [ ]:
tsRevWiki.shape

Let's create a timeseries that shows the number of edits by year. We will be resampling using the frequncy 'A' (see table above). Remember that we did this step with a `for` loop earlier in the notebook and got these results:

```
2003 10
2004 24
2005 55
2006 436
2007 638
2008 481
2009 371
2010 301
2011 168
2012 134
2013 138
2014 106
2015 152
2016 95
2017 159
```

In [ ]:
tsRevWiki.resample('YE').count()

### Visualizing a timeseries

pandas knows how to plot timeseries automatically, but let's get matplotlib in the namespace, given that it's needed to show the plots.

In [ ]:
import matplotlib.pyplot as plt
%matplotlib inline
plt.style.use('ggplot')

Let's store the result into a new series, so that we can call the plot method on it. It's as simple as that:

In [ ]:
revByYear = tsRevWiki.resample('YE').count()
revByYear.plot(figsize=(8,5), title="Number of revisions by year")
plt.show()

In [ ]:
revByMonth = tsRevWiki.resample('ME').count()
revByMonth.plot(figsize=(8,5), title="Number of revisions by month")
plt.show()

We can always **smooth** the data by making the interval bigger, for example, 3 months:

In [ ]:
revByThreeMonths = tsRevWiki.resample('3ME').count()
revByThreeMonths.plot(figsize=(8,5), title="Number of revisions quarterly")
plt.show()

We can focus on a single year, for example, 2007:

In [ ]:
revsIn2007 = tsRevWiki['2007']
revsIn2007Months = revsIn2007.resample('ME').count()
revsIn2007Months.plot(title="Number of revisions in 2007", # not using figsize
                      x_compat=True # this parameter suppresses how often the xticks are displayed
                     ) 
plt.show()

The same way, we can zoom in in the month of October 2017:

In [ ]:
revsOct17 = tsRevWiki['2017-10']
revsOct17Day = revsOct17.resample('D').count()
revsOct17Day.plot(figsize=(8,5), 
                  title="Number of revisions in 10/2017",
                  x_compat=True)
plt.show()

### Find most active editing days

Given that we can create a series based on daily edit counts, we can find the day with most edits:

In [ ]:
revByDays = tsRevWiki.resample('D').count()
revByDays.sort_values(ascending=False).head(10)

We can see how 2017/10/13, one of the days in the evolving Harvey Weinstein story, has the second largest number of edits.

### Find most active editors

In order to find editors, we will create a dataframe from the series, in order to use the method `groupby` which works better on dataframes.

Notice that we can create the dataframe using the columns of the timeseries tsRevWiki.

In [ ]:
dfWiki = DataFrame({'editors': tsRevWiki}, 
                     index=tsRevWiki.index)
dfWiki.head()

In [ ]:
dfWiki.shape

Now we can use the method `groupby` that will group together values in the column 'editors', find how often each editor occurs and show that value in a new column, "total":

In [ ]:
dfWikiT = dfWiki.groupby('editors').size().reset_index(name="total")
dfWikiT.head(10)

In [ ]:
dfWikiT.shape

We can see that the number of rows in this new dataframe is smaller, since some editors have more than one edit.

By the way, editor names such as 108.2.173.243, refer to the IP address of an editor. If someone edits a Wiki page without using an account, the system automatically captures their IP address.

Let's sort to find the most prolific editors:

In [ ]:
dfWikiT.sort_values('total', ascending=False)[:10]

### Focus on one editor

Let's look at the behavior of a single editor, for example, Nymf (third most active):

In [ ]:
oneUser = dfWiki[dfWiki['editors']=='Nymf'] # select from the frame the rows that fulfill the query
oneUser.head()

We can resample the events by year and plot the timeseries:

In [ ]:
oneUser.resample('YE').count().plot(legend=False, title="One user's Wiki editing activity")
plt.show()

Similarly, we can do this with grouping by year:

In [ ]:
oneUser.groupby(oneUser.index.year).count()

This shows that this user was active on 7 different years.

### Find page creator

Who created the page of Rose McGowan? We can find this info from the time series.

In [ ]:
dfWiki.tail(5)

In [ ]:
creator = dfWiki['editors'].iloc[-1]
creator

In [ ]:
dfWiki[dfWiki['editors']==creator]

This user only created the page and then never returned to make edits to it. Or, they created a username and then used that to do their edits.